# Azure DevOps Wiki Helpers

Azure DevOps Wiki sayfalarini okuma/olusturma/guncelleme icin ortak fonksiyonlar.
PAT, Databricks Secret Scope uzerinden okunur.

In [0]:
import requests
import base64
from urllib.parse import quote


organization = "ozanozeer"
project = "aXet Project"
wiki = "aXet-Project.wiki"


pat = dbutils.secrets.get(
    scope="sql-ozoezer",
    key="devopspac"
)


auth = base64.b64encode(
    f":{pat}".encode()
).decode()


headers = {
    "Authorization": f"Basic {auth}",
    "Content-Type": "application/json"
}


def get_wiki_page(page_path):

    project_encoded = quote(project)

    url = (
        f"https://dev.azure.com/{organization}/"
        f"{project_encoded}/_apis/wiki/wikis/"
        f"{wiki}/pages"
        f"?path={page_path}"
        f"&api-version=7.1"
    )

    response = requests.get(
        url,
        headers=headers
    )

    return response


def create_wiki_page(page_path, content):

    project_encoded = quote(project)

    url = (
        f"https://dev.azure.com/{organization}/"
        f"{project_encoded}/_apis/wiki/wikis/"
        f"{wiki}/pages"
        f"?path={page_path}"
        f"&api-version=7.1"
    )

    response = requests.put(
        url,
        headers=headers,
        json={
            "content": content
        }
    )

    return response


def update_wiki_page(page_path, content, version):

    project_encoded = quote(project)

    url = (
        f"https://dev.azure.com/{organization}/"
        f"{project_encoded}/_apis/wiki/wikis/"
        f"{wiki}/pages"
        f"?path={page_path}"
        f"&api-version=7.1"
    )

    headers_update = headers.copy()

    headers_update["If-Match"] = version

    response = requests.put(
        url,
        headers=headers_update,
        json={
            "content": content
        }
    )

    return response


def push_wiki_page(page_path, content):

    existing_page = get_wiki_page(page_path)

    if existing_page.status_code == 200:

        # Azure DevOps ETag header'dan alınır
        version = existing_page.headers.get("ETag")

        if version is None:
            print("ETag bulunamadı")
            print(existing_page.headers)
            return existing_page

        response = update_wiki_page(
            page_path,
            content,
            version
        )

        print("Wiki page updated")

    else:

        response = create_wiki_page(
            page_path,
            content
        )

        print("Wiki page created")


    print(response.status_code)
    print(response.text)

    return response

# Data Lake I/O Helpers

ADLS/abfss uzerinde okuma, yazma ve boyut hesaplama icin ortak fonksiyonlar.

In [0]:
def write_to_datalake(df_spark, path, mode="overwrite", file_format="json", num_files=1):

    writer = df_spark.coalesce(num_files).write.mode(mode)

    if file_format == "json":
        writer.json(path)
    elif file_format == "parquet":
        writer.parquet(path)
    elif file_format == "delta":
        writer.format("delta").save(path)
    else:
        raise ValueError(f"Desteklenmeyen format: {file_format}")

    print(f"Yazıldı: {path} ({file_format}, mode={mode})")

In [0]:
def read_from_datalake(path, file_format="json"):

    if file_format == "json":
        return spark.read.json(path)
    elif file_format == "parquet":
        return spark.read.parquet(path)
    elif file_format == "delta":
        return spark.read.format("delta").load(path)
    else:
        raise ValueError(f"Desteklenmeyen format: {file_format}")

In [ ]:
def get_datalake_size_bytes(path):

    total_size = 0

    for file_info in dbutils.fs.ls(path):
        if not file_info.isDir():
            total_size += file_info.size

    return total_size

# Open Data Portal Helpers (CKAN)

CKAN tabanli acik veri portallarindan (orn. acikveri.nilufer.bel.tr) veri seti
kaynagi bulma ve dosya indirme icin genel amacli fonksiyonlar.

In [ ]:
def get_ckan_resource(base_url, dataset_id, request_headers, preferred_format="XLSX"):

    url = f"{base_url}/api/3/action/package_show"

    response = requests.get(
        url,
        params={"id": dataset_id},
        headers=request_headers,
        timeout=30,
    )
    response.raise_for_status()

    resources = response.json()["result"]["resources"]

    for resource in resources:
        if resource.get("format", "").upper() == preferred_format:
            return resource

    return resources[0] if resources else None

In [ ]:
def download_file(url, dest_path, request_headers=None, chunk_size=1024 * 64):

    with requests.get(url, headers=request_headers, stream=True, timeout=60) as response:
        response.raise_for_status()

        with open(dest_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                f.write(chunk)

    return dest_path

# Agent-Friendly Wiki Content Builders

Bir dataset'in metadata + schema + ornek verisini, bir AI agent'in kolayca
parse edebilecegi yapilandirilmis (JSON blogu + markdown tablo) formatta
wiki icerigine ceviren fonksiyonlar.

In [ ]:
import json
from datetime import datetime, timezone


def build_agent_friendly_wiki_content(
    df_spark,
    dataset_name,
    source_url,
    license_name,
    datalake_path,
    sample_size=10,
):

    size_bytes = get_datalake_size_bytes(datalake_path)
    partition_count = df_spark.rdd.getNumPartitions()

    schema_fields = [
        {"name": field.name, "type": field.dataType.simpleString()}
        for field in df_spark.schema.fields
    ]

    metadata = {
        "dataset_name": dataset_name,
        "source_url": source_url,
        "license": license_name,
        "last_updated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "size_bytes": size_bytes,
        "partition_count": partition_count,
        "columns": schema_fields,
    }

    metadata_json = json.dumps(metadata, ensure_ascii=False, indent=2)

    schema_markdown = "\n".join(
        [
            f"| {field['name']} | {field['type']} |"
            for field in schema_fields
        ]
    )

    sample_json = df_spark.limit(sample_size).toPandas().to_json(orient="records", force_ascii=False)
    sample_pretty = json.dumps(json.loads(sample_json), ensure_ascii=False, indent=2)

    content = f"""# {dataset_name}

## Metadata

```json
{metadata_json}
```

## Schema

| Column | Type |
|---|---|
{schema_markdown}

## Sample Data (first {sample_size} rows, JSON)

```json
{sample_pretty}
```
"""

    return content